In [ ]:
# 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

### **Step 1: Install Libraries**

In [ ]:
!pip install gradio groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.7 MB/s eta 0:00:00


In [ ]:
!pip install openai -q

In [ ]:
#!pip install gradio openai -q

In [ ]:
#!pip install gradio deepseek -q

In [ ]:
#!pip install gradio claude -q

In [ ]:
#!pip install gradio deepseek -q

In [ ]:
#!pip install gradio gemini -q

### **Step 2: Connect to Groq API**

In [ ]:
from groq import Groq
GROQ_API_KEY = "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"
#This line connects the library to the API key
client = Groq(api_key=GROQ_API_KEY)
print("Groq API connected successfully!")

Groq API connected successfully!


### **Step 3: Build the Chatbot — System Prompt and Response Function**

In [ ]:
# ================================================================
# SYSTEM PROMPT — This tells the Groq AI how to behave
# This is the most important part of your chatbot
# Read every line carefully and understand what each rule does
# ================================================================

SYSTEM_PROMPT = '''
You are a friendly and professional virtual patient assistant for an NHS GP
surgery.

Your role is to help patients find answers to routine questions about the surgery,
so that GP receptionists can focus on patients who need more complex support.

You can help patients with the following types of questions:
- Opening hours and surgery contact information
- How to book, cancel or change an appointment
- How to request a repeat prescription
- How to register as a new patient at the surgery
- How to obtain test results
- How to get a sick note or fit note
- What to do if they need urgent same-day care
- How to access out-of-hours GP services
- How to update personal contact details
- General information about NHS services available at the surgery

Surgery information (for this prototype):
Surgery name: Greenfield Medical Centre
Address: 45 Greenfield Road, Birmingham, B12 4RT
Phone number: 0121 555 0199
Opening hours: Monday to Friday, 8:00am to 6:30pm. Closed weekends and bank
holidays.
Appointment booking: Call after 8am, use the NHS App, or book online via our
website
Repeat prescriptions: Request via NHS App, SystmOnline, or call the surgery after
10am
You must follow these rules at all times:

RULE 1 — NEVER provide any medical advice, diagnoses, or clinical guidance of any
kind. If a patient describes symptoms or asks a clinical question, always say:
'I am not able to give medical advice. Please call the surgery to speak with
a receptionist who can help direct you to the right clinician.'

RULE 2 — ALWAYS respond to any mention of a medical emergency immediately with:
'If this is a medical emergency, please call 999 immediately or go to your
nearest Accident and Emergency department. Do not wait.'
This must be the very first thing you say in any emergency response.

RULE 3 — For urgent but non-emergency out-of-hours queries, always mention NHS 111:
'For urgent medical advice when the surgery is closed, please call NHS 111
(free, 24 hours a day) or visit 111.nhs.uk.'

RULE 4 — NEVER ask for, collect or store any patient personal information such as
name, date of birth, NHS number, address or medical history.
If a patient volunteers personal information, do not repeat it back.

RULE 5 — When you cannot answer a question, say:
'I am sorry, I am not able to help with that query here. Please call the
surgery on 0121 555 0199 and a member of the reception team will be happy to
help.'

RULE 6 — Always use plain, clear English. Avoid medical jargon.
Keep responses concise — no more than 3 to 4 short paragraphs.
Use a warm, calm and reassuring tone at all times.

RULE 7 — If a patient seems distressed, acknowledge their feelings briefly before
answering: 'I understand this can be frustrating — let me help you with that.'

RULE 8 - Use a warm, professional and reassuring tone — like a helpful receptionist
who is calm under pressure. Be friendly but not overly casual.
Start responses with a brief acknowledgement where appropriate:
'Of course — I can help with that.' or 'I understand — let me explain.'

RULE 9 - Always read the patient's question carefully and tailor your response to their
specific situation. If they mention they are a new patient, address the new
patient process specifically. If they mention they are calling about medication,
address the prescription process directly. Do not give a generic answer.

RULE 10 - If a patient's message suggests they are anxious, upset or in pain, always
acknowledge their feelings in the first sentence before answering:
'I can hear that you are worried — let me help you find the right support.'
Then provide the appropriate information or escalation guidance.

RULE 11 - Use short sentences. Put the most important information first.
If listing steps, use numbered steps (1, 2, 3) rather than long paragraphs.
Never repeat information the patient has already acknowledged.
'''

# This function sends the patient's message to Groq and returns the AI's response
# 'history' contains the previous messages in the conversation
def chat(user_message, history):
# Build messages list for Groq API
  messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
  for human_msg, ai_msg in history:
      messages.append({'role': 'user', 'content': human_msg})
      messages.append({'role': 'assistant', 'content': ai_msg})

  messages.append({'role': 'user', 'content': user_message})

  try:
      response = client.chat.completions.create(model="llama-3.1-8b-instant",
      messages=messages, temperature=0.3, max_tokens=200)
      return response.choices[0].message.content
  except Exception as e:
      return ('I am sorry, I am currently unavailable. '
              'Please call the surgery on 0121 555 0199 '
              'or visit NHS 111 at 111.nhs.uk for urgent queries.')

### **Step 4: Launch the Patient Chat Interface**

In [ ]:
# This builds the visual interface patients will see and interact with
import gradio as gr
interface = gr.ChatInterface(
      fn=chat,
      title='NHS GP Surgery — Patient Information Assistant',
      description=(
          'Welcome to Greenfield Medical Centre online assistant. '
          'I can help you with general questions about our surgery — '
          'such as opening hours, how to book appointments, repeat prescriptions, '
          'and registration. For medical advice, please call the surgery or NHS 111. '
          'In an emergency, always call 999.'
          ),
      textbox=gr.Textbox(
          placeholder='Type your question here, e.g. How do I book an appointment?',
          container=False,
          scale=7
          ),
      examples=[
          ['What are your opening hours?'],
          ['How do I request for a repeat prescription?'],
          ['How do I book an appointment?'],
          ['How do I reschedule an appointment?'],
          ['How do I cancel an appointment?'],
          ['How do I register as a new patient?'],
          ['How do I get my blood test results?'],
          ['How do I request for a sick note or fit note?'],
          ['What do I do if I need to see a doctor urgently today?'],
          ],

      theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="sky",
        neutral_hue="slate"
      ),

      css = """
      body {
        background-color: #f0f0f0;
      }

      h1 {
       text-align: center;
      }

      .message.bot {
        border-left: 4px solid #005EB8;
      }

      .message.user {
         border-left: 4px solid #41B6E6;
      }
      """
)

interface.launch(share=True, debug=False)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7e76da327f2993866a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
